# Cryptocurrency Web Scraping with Selenium — CoinGecko

# https://www.coingecko.com/

In [1]:
%pip install selenium pandas

Note: you may need to restart the kernel to use updated packages.


In [1]:
import time
import re
import os
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

## 1. Open CoinGecko

In [2]:
options = Options()
options.add_argument('--start-maximized')
options.page_load_strategy = 'eager'

driver = webdriver.Chrome(options=options)
driver.set_page_load_timeout(60)
driver.get('https://www.coingecko.com/')
time.sleep(8)

print('Page title:', driver.title)
print('Current URL:', driver.current_url)
print('Page loaded.')

Page title: Cryptocurrency Prices, Charts, and Crypto Market Cap | CoinGecko
Current URL: https://www.coingecko.com/
Page loaded.


## 2. Scroll the page to load more rows

In [3]:
for _ in range(50):
    driver.execute_script('window.scrollBy(0, 2000);')
    time.sleep(3)
driver.execute_script('window.scrollTo(0, 0);')
time.sleep(2)

## 3. Extract visible table rows

In [4]:
rows = driver.find_elements(By.XPATH, '//table//tbody/tr')
print('Rows found:', len(rows))

data = []
for row in rows:
    cells = row.find_elements(By.TAG_NAME, 'td')
    values = [cell.text for cell in cells]
    if values:
        data.append(values)
print('Records extracted:', len(data))

Rows found: 100
Records extracted: 100


In [5]:
from selenium.webdriver.common.by import By
import pandas as pd
import time

all_data = []

for page in range(1, 31):  # Pages 1 to 10
    url = f"https://www.coingecko.com/en?page={page}"
    driver.get(url)

    time.sleep(5)

    rows = driver.find_elements(By.XPATH, "//table//tbody/tr")
    print(f"Page {page}: {len(rows)} rows")

    for row in rows:
        cells = row.find_elements(By.TAG_NAME, "td")
        values = [cell.text for cell in cells]

        if values:
            all_data.append(values)

print("Total records extracted:", len(all_data))

Page 1: 100 rows
Page 2: 100 rows
Page 3: 100 rows
Page 4: 100 rows
Page 5: 100 rows
Page 6: 100 rows
Page 7: 100 rows
Page 8: 100 rows
Page 9: 100 rows
Page 10: 100 rows
Page 11: 100 rows
Page 12: 100 rows
Page 13: 100 rows
Page 14: 100 rows
Page 15: 100 rows
Page 16: 100 rows
Page 17: 100 rows
Page 18: 100 rows
Page 19: 100 rows
Page 20: 100 rows
Page 21: 100 rows
Page 22: 100 rows
Page 23: 100 rows
Page 24: 100 rows
Page 25: 100 rows
Page 26: 100 rows
Page 27: 100 rows
Page 28: 100 rows
Page 29: 100 rows
Page 30: 100 rows
Total records extracted: 3000


## 4. Create a DataFrame with generic column names

In [6]:
if not all_data:
    print('No table rows found. The page layout may have changed or the table did not load.')
else:
    max_cols = max(len(row) for row in all_data)
    data_padded = [row + [''] * (max_cols - len(row)) for row in all_data]
    df = pd.DataFrame(data_padded, columns=[f'Column_{i}' for i in range(1, max_cols + 1)])
    display(df.head(10))
    print('DataFrame shape:', df.shape)

,Column_1,Column_2,Column_3,Column_4,Column_5,Column_6,Column_7,Column_8,Column_9,Column_10,Column_11,Column_12,Column_13,Column_14
0,,1,Bitcoin BTC,Buy,"$81,280.89",0.6%,3.1%,2.2%,,"$41,441,960,779","$1,633,736,485,222",,,
1,,2,Ethereum ETH,Buy,"$2,372.42",0.7%,5.9%,8.0%,,"$18,921,242,787","$289,740,969,837",,,
2,,3,Tether USDT,,$0.9991,0.0%,0.0%,0.0%,,"$72,465,083,704","$183,876,315,874",,,
3,,4,BNB BNB,Buy,$719.06,0.4%,3.5%,6.6%,,"$942,036,953","$95,751,544,591",,,
4,,5,XRP XRP,Buy,$1.31,0.2%,4.3%,6.7%,,"$2,332,257,331","$82,531,684,555",,,
5,,6,USDC USDC,,$0.9996,0.0%,0.0%,0.0%,,"$21,236,233,929","$72,700,046,889",,,
6,,7,Solana SOL,Buy,$102.75,0.8%,4.8%,11.2%,,"$4,147,550,745","$60,552,024,064",,,
7,,8,TRON TRX,Buy,$0.3321,0.3%,0.1%,1.2%,,"$503,828,510","$31,550,699,326",,,
8,,9,Figure Heloc FIGR_HELOC,Buy,$1.04,0.0%,1.3%,2.1%,,"$42,996,287","$24,781,146,821",,,
9,,10,Zcash ZEC,Buy,"$1,101.18",0.9%,12.4%,11.8%,,"$1,112,973,893","$18,701,484,952",,,


DataFrame shape: (3000, 14)


## 5. Clean text using Regex

In [7]:
if 'df' in globals():
    for col in df.columns:
        df[col] = df[col].astype(str).apply(lambda x: re.sub(r'\s+', ' ', x).strip())
    display(df.head(10))

,Column_1,Column_2,Column_3,Column_4,Column_5,Column_6,Column_7,Column_8,Column_9,Column_10,Column_11,Column_12,Column_13,Column_14
0,,1,Bitcoin BTC,Buy,"$81,280.89",0.6%,3.1%,2.2%,,"$41,441,960,779","$1,633,736,485,222",,,
1,,2,Ethereum ETH,Buy,"$2,372.42",0.7%,5.9%,8.0%,,"$18,921,242,787","$289,740,969,837",,,
2,,3,Tether USDT,,$0.9991,0.0%,0.0%,0.0%,,"$72,465,083,704","$183,876,315,874",,,
3,,4,BNB BNB,Buy,$719.06,0.4%,3.5%,6.6%,,"$942,036,953","$95,751,544,591",,,
4,,5,XRP XRP,Buy,$1.31,0.2%,4.3%,6.7%,,"$2,332,257,331","$82,531,684,555",,,
5,,6,USDC USDC,,$0.9996,0.0%,0.0%,0.0%,,"$21,236,233,929","$72,700,046,889",,,
6,,7,Solana SOL,Buy,$102.75,0.8%,4.8%,11.2%,,"$4,147,550,745","$60,552,024,064",,,
7,,8,TRON TRX,Buy,$0.3321,0.3%,0.1%,1.2%,,"$503,828,510","$31,550,699,326",,,
8,,9,Figure Heloc FIGR_HELOC,Buy,$1.04,0.0%,1.3%,2.1%,,"$42,996,287","$24,781,146,821",,,
9,,10,Zcash ZEC,Buy,"$1,101.18",0.9%,12.4%,11.8%,,"$1,112,973,893","$18,701,484,952",,,


In [8]:
# Drop empty columns
df = df.drop(columns=['Column_1', 'Column_12', 'Column_13', 'Column_14'],
             errors='ignore')

# Reset index
df = df.reset_index(drop=True)

# Rename columns
df.columns = [
    "Rank",
    "Name",
    "Buy",
    "Price",
    "1h %",
    "24h %",
    "7d %",
    "24h Volume",
    "Market Cap",
    "Circulating Supply"
]

df.head()

,Rank,Name,Buy,Price,1h %,24h %,7d %,24h Volume,Market Cap,Circulating Supply
0,1,Bitcoin BTC,Buy,"$81,280.89",0.6%,3.1%,2.2%,,"$41,441,960,779","$1,633,736,485,222"
1,2,Ethereum ETH,Buy,"$2,372.42",0.7%,5.9%,8.0%,,"$18,921,242,787","$289,740,969,837"
2,3,Tether USDT,,$0.9991,0.0%,0.0%,0.0%,,"$72,465,083,704","$183,876,315,874"
3,4,BNB BNB,Buy,$719.06,0.4%,3.5%,6.6%,,"$942,036,953","$95,751,544,591"
4,5,XRP XRP,Buy,$1.31,0.2%,4.3%,6.7%,,"$2,332,257,331","$82,531,684,555"


In [9]:
import numpy as np

df = df.replace(r'^\s*$', np.nan, regex=True)


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_8108\1811326959.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(r'^\s*$', np.nan, regex=True)


In [10]:
df.isnull().sum()

Rank                     0
Name                     0
Buy                    570
Price                    0
1h %                     0
24h %                    0
7d %                     0
24h Volume            3000
Market Cap               0
Circulating Supply       0
dtype: int64

In [11]:
df.drop(columns=["24h Volume"], inplace=True)

In [12]:
df

,Rank,Name,Buy,Price,1h %,24h %,7d %,Market Cap,Circulating Supply
0,1,Bitcoin BTC,Buy,"$81,280.89",0.6%,3.1%,2.2%,"$41,441,960,779","$1,633,736,485,222"
1,2,Ethereum ETH,Buy,"$2,372.42",0.7%,5.9%,8.0%,"$18,921,242,787","$289,740,969,837"
2,3,Tether USDT,NaN,$0.9991,0.0%,0.0%,0.0%,"$72,465,083,704","$183,876,315,874"
3,4,BNB BNB,Buy,$719.06,0.4%,3.5%,6.6%,"$942,036,953","$95,751,544,591"
4,5,XRP XRP,Buy,$1.31,0.2%,4.3%,6.7%,"$2,332,257,331","$82,531,684,555"
...,...,...,...,...,...,...,...,...,...
2995,96,Pyth Network PYTH,Buy,$0.08727,2.4%,9.6%,17.7%,"$103,917,754","$687,225,345"
2996,97,Aptos APT,Buy,$0.7765,0.6%,5.7%,1.0%,"$71,207,434","$685,610,026"
2997,98,XDC Network XDC,Buy,$0.03386,1.0%,3.1%,4.2%,"$13,399,709","$675,418,725"
2998,99,Beldex BDX,Buy,$0.08395,1.1%,1.7%,12.6%,"$2,558,632","$660,781,986"


## 6. Save the scraped data as CSV

In [13]:
df.to_csv('coingecko_data4.csv', index=False, encoding='utf-8-sig')
print('Saved as coingecko_data.csv')

Saved as coingecko_data.csv


## 7. Close the browser

In [14]:
driver.quit()
print('Browser closed.')

Browser closed.
